# A4 — Ultralytics YOLO Experiment Notebook

This notebook is the **primary interface** of an A4 project (see the repository `README.md`).
It is a guided, executable experiment that leads through the full lifecycle — setup,
dataset validation, training, evaluation, summary, export, and report — while delegating
all core ML functionality to **Ultralytics** (package + Platform).

**How to use this template**
- Copy this notebook into `projects/<Task>/<Dataset>/notebook.ipynb`.
- Fill in the marked configuration cells (task, model, dataset, experiment name).
- Put dataset-specific YAML under `configs/datasets/`, and any *intentional* overrides
  under `configs/experiments/<experiment_name>.yaml`.
- Run cells top to bottom. Custom code should stay rare and minimal — if Ultralytics or
  Ultralytics Platform already does it, use it (see the repository's `yolo-*` skills).
- All Ultralytics-generated outputs live under `runs/` — this template does not
  duplicate them into a separate `artifacts/`, `checkpoints/`, or `logs/` structure.

## 0. Environment

Install/upgrade the package, confirm the environment, and configure the two settings
that most often break notebooks on fresh Colab/Kaggle runtimes: relative dataset paths
resolving against the wrong `datasets_dir`, and (optionally) Platform credentials.

In [ ]:
!git clone https://github.com/AmirMahdiRezaeiEECS/ALL-IN-ONE-VISION-4
%cd ALL-IN-ONE-VISION-4

In [ ]:
%pip install -q -U ultralytics

In [ ]:
import os
from pathlib import Path

import yaml

from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
print(f"Working directory: {PROJECT_ROOT}")

In [ ]:
!yolo checks

**Optional — Ultralytics Platform.** Platform (not a bespoke tracker) is this repo's
primary tool for dataset/experiment/model management. Set an API key to enable
`data=ul://username/datasets/dataset-slug` and `project=username/project-slug`
(local training then streams metrics to a Platform project). Leave unset for a
purely local run — nothing else in this notebook depends on it.

In [ ]:
# os.environ["ULTRALYTICS_API_KEY"] = "..."  # uncomment to enable Platform streaming

# If `path:` in your data.yaml is relative, it resolves against `datasets_dir` below.
# Point it at your actual dataset root once per environment (Colab/Kaggle sessions reset).
# !yolo settings datasets_dir=/content/datasets

## 1. Experiment Setup

Define the experiment identity and select only the parameters that need to change.
Ultralytics' defaults remain the baseline — see `yolo cfg` for every default argument.
An optional `configs/experiments/<EXPERIMENT_NAME>.yaml` supplies intentional overrides
(e.g. `epochs`, `imgsz`, augmentation) without hard-coding them in the notebook.

In [ ]:
from ultralytics.data.split_dota import split_test, split_trainval

# Split train and val set, with labels.
split_trainval(
    data_root="path/to/DOTAv1.0/",
    save_dir="path/to/DOTAv1.0-split/",
    rates=[0.5, 1.0, 1.5],  # multiscale
    gap=500,
)
# Split test set, without labels.
split_test(
    data_root="path/to/DOTAv1.0/",
    save_dir="path/to/DOTAv1.0-split/",
    rates=[0.5, 1.0, 1.5],  # multiscale
    gap=500,
)

In [ ]:
# --- Experiment identity -----------------------------------------------------
TASK = "obb"  # detect | segment | semantic | depth | classify | pose | obb
MODEL = "yolo26n-obb.pt"  # ALWAYS a pretrained checkpoint; start with 'n' to validate the
                        # pipeline cheaply, then scale up (see the yolo-models skill)
DATA = "configs/datasets/DOTAv1.yaml"  # local yaml, classify folder, or ul://... URI

EXPERIMENT_NAME = "Obb_experiment"  # self-describing, e.g. "0906_yolo26n_voc_e100"
PROJECT = f"runs/{TASK}"       # or "username/project-slug" to stream to Platform

# --- Config-driven overrides ---------------------------------------------------
# Only settings we have a strong reason to change belong here (README: "Default-First
# Configuration"). Everything else is left to Ultralytics.
experiment_config_path = Path(f"configs/experiments/{EXPERIMENT_NAME}.yaml")
overrides = {}
if experiment_config_path.exists():
    with open(experiment_config_path) as f:
        overrides = yaml.safe_load(f) or {}

print(f"Task:      {TASK}")
print(f"Model:     {MODEL}")
print(f"Data:      {DATA}")
print(f"Run:       {PROJECT}/{EXPERIMENT_NAME}")
print(f"Overrides: {overrides}")

**Sanity check** — the task/model suffix mismatch is a common, silent footgun
(e.g. training `-seg` data with a plain detect checkpoint yields mAP ≈ 0). This only
warns; it does not block families with different naming (YOLO-World, YOLOE, SAM, RT-DETR).

In [ ]:
_TASK_SUFFIX = {
    "detect": "", "segment": "-seg", "semantic": "-sem", "depth": "-depth",
    "classify": "-cls", "pose": "-pose", "obb": "-obb",
}
_stem = Path(MODEL).stem
_all_suffixes = [s for s in _TASK_SUFFIX.values() if s]
_expected = _TASK_SUFFIX.get(TASK, "")

if _stem.startswith(("yolo",)):  # only applies to plain YOLO family naming
    if _expected and not _stem.endswith(_expected):
        print(f"WARNING: MODEL='{MODEL}' does not end in '{_expected}' for TASK='{TASK}'.")
    elif not _expected and any(_stem.endswith(s) for s in _all_suffixes):
        print(f"WARNING: MODEL='{MODEL}' looks task-suffixed but TASK='{TASK}' (detect) was chosen.")

## 2. Dataset Preparation & Validation

The #1 cause of silent training failure is a malformed dataset. This follows the
`yolo-datasets` skill's validation order exactly — no hand-rolled label parsing, because
the smoke test below already produces the real distribution and augmentation plots
(`labels.jpg`, `labels_correlogram.jpg`, `train_batch0.jpg`) via Ultralytics itself.
If raw annotations still need converting (COCO/DOTA/masks), use the built-in converters
in `ultralytics.data.converter` (see the `yolo-datasets` skill); keep any one-off
conversion script under `scripts/data/` rather than inline here.

In [ ]:
from ultralytics.data.utils import check_det_dataset

if TASK != "classify":
    dataset_info = check_det_dataset(DATA)  # validates data.yaml, resolves paths
    names = dataset_info["names"]
    print(f"Classes ({len(names)}): {names}")
else:
    dataset_info = None
    names = None
    print(f"Classification dataset folder: {DATA}")

**Visual spot check** (detect only — five-column `class cx cy w h` rows).
Uses Ultralytics' own `img2label_paths` for the images→labels mirror, rather than
a hand-rolled string replace — the mirror rule replaces the *last* `/images/` segment,
which a naive `.replace()` gets wrong on paths containing "images" more than once.

In [ ]:
from ultralytics.data.utils import visualize_image_annotations
from ultralytics.data.utils import img2label_paths

if TASK == "detect":
    train_dir = dataset_info["train"]
    train_dir = Path(train_dir[0] if isinstance(train_dir, list) else train_dir)
    sample_image = next(
        (p for ext in ("*.jpg", "*.jpeg", "*.png") for p in train_dir.glob(ext)), None
    )
    sample_label = Path(img2label_paths([str(sample_image)])[0])
    visualize_image_annotations(str(sample_image), str(sample_label), label_map=names)

**Task-loader smoke test** — builds the *real* dataset/dataloader on a small fraction,
which is what actually surfaces label/format problems (not a custom pre-check).
Set `RUN_SMOKE_TEST = True` once per new dataset and inspect the outputs below before
committing to a full run.

In [ ]:
RUN_SMOKE_TEST = False
SMOKE_DIR = Path(PROJECT) / "smoke_test"

if RUN_SMOKE_TEST:
    YOLO(MODEL).train(
        data=DATA,
        epochs=1,
        fraction=0.1,
        project=PROJECT,
        name="smoke_test",
        exist_ok=True,
    )

In [ ]:
from IPython.display import Image, display

if RUN_SMOKE_TEST:
    # Class balance, aug-space coverage, and label correctness — all from Ultralytics,
    # none of it hand-rolled.
    for plot_name in ("labels.jpg", "labels_correlogram.jpg", "train_batch0.jpg"):
        plot_path = SMOKE_DIR / plot_name
        if plot_path.exists():
            display(Image(filename=str(plot_path)))

## 3. Model & Training

Load the pretrained checkpoint and train. Class-count changes are handled automatically
by Ultralytics — never set `pretrained=False` outside explicit research (see README Note 1:
*do not throw away pretrained knowledge*).

In [ ]:
model = YOLO(MODEL)  # always start from pretrained weights

model.train(
    data=DATA,
    project=PROJECT,
    name=EXPERIMENT_NAME,
    **overrides,
)

RUN_DIR = Path(model.trainer.save_dir)
print(f"Run saved to: {RUN_DIR}")

**Optional — hyperparameter tuning.** Per the `yolo-tuning` skill, tuning is the
*last* lever, worth roughly 0.5–2 mAP once data quality, training length, input size,
model size, and augmentation are already exhausted. Leave `RUN_TUNE = False` for a
first pass; only enable it after inspecting Section 4 and concluding the baseline
model/data setup itself is sound.

In [ ]:
RUN_TUNE = False

if RUN_TUNE:
    tune_model = YOLO(MODEL)
    tune_model.tune(
        data=DATA,
        epochs=30,
        iterations=100,
        plots=False,
        save=False,
        val=False,
    )
    # Inspect runs/<task>/tune/best_hyperparameters.yaml, then retrain fully with it:
    # overrides.update(yaml.safe_load(open("runs/<task>/tune/best_hyperparameters.yaml")))

## 4. Evaluation & Inspection

Always evaluate `best.pt`, not `last.pt`.

In [ ]:
BEST = RUN_DIR / "weights" / "best.pt"
best_model = YOLO(BEST)

metrics = best_model.val(data=DATA, plots=True, save_json=True)
print(metrics.results_dict)

Inspect the standard Ultralytics plots for this run.

In [ ]:
for plot_name in (
    "labels.jpg",
    "results.png",
    "confusion_matrix.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
):
    plot_path = RUN_DIR / plot_name
    if plot_path.exists():
        display(Image(filename=str(plot_path)))

Spot-check predictions on a handful of validation images.

In [ ]:
import cv2
from PIL import Image as PILImage

val_dir = dataset_info["val"] if dataset_info else DATA
val_dir = val_dir[0] if isinstance(val_dir, list) else val_dir

predictions = best_model.predict(source=val_dir, conf=0.25, verbose=False)
for r in predictions[:5]:
    display(PILImage.fromarray(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)))

## 5. Experiment Summary

Consolidate the configuration and results Ultralytics already recorded
(`args.yaml`, `results.csv`) into a concise record — no separate tracking system needed.
Reads back from `RUN_DIR` rather than relying only on in-memory state, so this cell
still works if it's re-run on its own later.

In [ ]:
import pandas as pd

run_args = yaml.safe_load(open(RUN_DIR / "args.yaml"))
results_csv = pd.read_csv(RUN_DIR / "results.csv")
results_csv.columns = [c.strip() for c in results_csv.columns]

if "metrics" not in dir():
    best_model = YOLO(RUN_DIR / "weights" / "best.pt")
    metrics = best_model.val(data=DATA)

summary = {
    "experiment": EXPERIMENT_NAME,
    "task": TASK,
    "model": MODEL,
    "data": DATA,
    "epochs_ran": int(results_csv["epoch"].iloc[-1]) + 1,
    "overrides": overrides,
    "final_metrics": metrics.results_dict,
    "run_dir": str(RUN_DIR),
}

for key, value in summary.items():
    print(f"{key}: {value}")

## 6. Export

Export to the deployment format and verify parity against the `.pt` baseline
(see the `yolo-export` skill's format matrix for the full list of 20 formats and args).

In [ ]:
EXPORT_FORMAT = "onnx"  # torchscript | onnx | openvino | engine | coreml | ...

export_path = best_model.export(format=EXPORT_FORMAT)
print(f"Exported to: {export_path}")

In [ ]:
exported_model = YOLO(export_path)
exported_metrics = exported_model.val(data=DATA).results_dict

print("Baseline (.pt):", summary["final_metrics"])
print("Exported:      ", exported_metrics)

## 7. Report

Write a short, reproducible Markdown report into `docs/` — dataset, configuration,
training results, evaluation, and exported artifacts in one place.

In [ ]:
report_path = Path("docs") / f"{EXPERIMENT_NAME}_report.md"
report_path.parent.mkdir(parents=True, exist_ok=True)

report = f"""# Experiment Report — {EXPERIMENT_NAME}

## Configuration
- Task: {TASK}
- Model: {MODEL}
- Data: {DATA}
- Overrides: {overrides}

## Training
- Run directory: `{RUN_DIR}`
- Epochs ran: {summary['epochs_ran']}

## Evaluation
- Final metrics: {summary['final_metrics']}

## Export
- Format: {EXPORT_FORMAT}
- Artifact: `{export_path}`
- Exported metrics: {exported_metrics}

## Notes
_Add qualitative observations, failure cases, and next steps here._
"""

report_path.write_text(report)
print(f"Report written to {report_path}")

---
For details on any stage beyond this notebook's scope, consult the matching skill:
`yolo-models`, `yolo-datasets`, `yolo-training`, `yolo-tuning`, `yolo-inference`, `yolo-export`.